# Can FAOSTAT crop statistics detect weather shocks in Uganda?
**Data:** FAOSTAT crop production, area harvested and yield for Uganda, 1990–2024 (bulk release of December 2025), with FAO's data-quality flags
**Purpose:** before linking rainfall to harvests, check whether the national crop statistics actually record year-to-year changes in harvests, or mainly reflect projections and changes in method.

**Why this matters.** FAOSTAT is the default source for country crop data, and is widely used to estimate how weather affects yields. If the series are projected forward between agricultural censuses, a rainfall–yield regression measures the statistical method rather than the harvest.

Each finding is written up in a markdown cell directly after the code and output that produced it.

## Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '../scripts')
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# House style, shared with the rainfall project
WET, DRY, ACCENT, INK, MUTED, GRID = '#2b6c9e', '#c4572e', '#16825a', '#132019', '#7a877f', '#e6e9e4'
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c9cfc8', 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.labelcolor': MUTED,
    'legend.frameon': False,
})
pd.set_option('display.width', 200)

CROPS = ['Maize (corn)', 'Beans, dry', 'Sorghum', 'Millet', 'Cassava, fresh', 'Sweet potatoes',
         'Plantains and cooking bananas', 'Groundnuts, excluding shelled', 'Rice', 'Potatoes',
         'Soya beans', 'Sesame seed', 'Coffee, green']
fao = pd.read_csv('../data/raw/faostat_crops_uganda.csv')
fao = fao[(fao['year'] >= 1990) & fao['item'].isin(CROPS)]
wide = {el: fao[fao['element'] == el].pivot(index='year', columns='item', values='value')[CROPS]
        for el in ['Production', 'Area harvested', 'Yield']}
print('Years:', fao['year'].min(), '–', fao['year'].max(), '| crops:', len(CROPS))

---
## 1. What the flags say

In [ ]:
flags = (fao[fao['element'] == 'Production'].groupby('item')['flag']
         .value_counts(normalize=True).unstack(fill_value=0).reindex(CROPS))
flag_names = {'A': 'Official', 'E': 'Estimated', 'I': 'Imputed', 'X': 'External'}
(100 * flags.rename(columns=flag_names)).round(0)

**Finding:** taken at face value, the data look reliable. For the main food crops, over 90% of production values are flagged **official** (reported by the government). The flags record who supplied a number, though, not how it was produced. Sections 2–4 test how much real variation the official numbers contain.

---
## 2. The signature of projected data

In [ ]:
growth = 100 * wide['Area harvested'].pct_change()
near_flat = (growth.abs() < 0.5)
same_yield = wide['Yield'].diff().abs() < 1e-9
periods = {'1991–2007': (1991, 2007), '2013–2015': (2013, 2015), '2016–2024': (2016, 2024)}
tab = pd.DataFrame({f'% of years with area change < 0.5%, {k}': 100 * near_flat.loc[a:b].mean() for k, (a, b) in periods.items()})
tab['median area change 2013–2015 (%)'] = growth.loc[2013:2015].abs().median()
tab.round(1)

In [ ]:
show = ['Maize (corn)', 'Beans, dry', 'Sorghum', 'Cassava, fresh', 'Plantains and cooking bananas']
fig, axes = plt.subplots(1, len(show), figsize=(13, 3.2), sharex=True)
for ax, c in zip(axes, show):
    s = wide['Area harvested'][c] / 1e3
    ax.plot(s.index, s.values, color=WET, lw=2, marker='o', ms=2.5)
    ax.axvspan(2012.5, 2015.5, color=GRID, alpha=.7, lw=0)
    ax.set_title(c.split(',')[0].split(' (')[0], fontsize=10)
axes[0].set_ylabel('Area harvested (000 ha)')
fig.suptitle('Area harvested barely moves in 2013–2015 (shaded), after a large break in 2008', x=0.01, ha='left', fontweight='bold')
plt.tight_layout(); plt.show()

**Finding:** in **2013–2015**, the area harvested for beans, sorghum, millet, cassava and sweet potatoes changed by less than half a percent in every one of the three years (median 0.0–0.3%), and maize and bananas by under 1%. Before 2008, a change this small happened in fewer than one year in five for any of these crops. Real harvested area swings with rainfall, prices and insecurity; series this smooth are projections, most likely from the 2008/09 Uganda Census of Agriculture baseline rolled forward. Those years carry essentially no information about weather.

---
## 3. Breaks from changes in method

In [ ]:
big = growth.loc[1991:2024].abs()
breaks = (wide['Production'].pct_change() * 100).loc[[2008, 2018, 2019]].T.round(0)
breaks.columns = [f'Production change {y} (%)' for y in breaks.columns]
breaks

**Finding:** 2008 is a break, not a harvest. In one year bananas fell 54%, millet 62% and cassava 42%, while maize rose 83% and beans 110%. No weather event moves crops in opposite directions by these amounts: 2008 is when the new census baseline replaced the old series. Smaller jumps around 2018–2019 (soya beans +284%, beans −54% then +80%) coincide with the introduction of UBOS's Annual Agricultural Survey.

Some long-run trends are also implausible. Cassava's reported yield falls from about 13.5 t/ha in the early 2000s to about 2 t/ha in 2024, an 85% collapse that agronomic evidence does not support.

---
## 4. Do the statistics respond to rainfall at all?

In [ ]:
from prices import seasonal_rain_z
mam = seasonal_rain_z([3, 4, 5]).xs('Uganda (national)')
ond = seasonal_rain_z([10, 11, 12]).xs('Uganda (national)')
rain_year = (mam + ond.shift(1)).rename('rain')   # first season this year + second season last year

log_change = np.log(wide['Production']).diff()
rows = []
for c in ['Maize (corn)', 'Beans, dry', 'Sorghum', 'Millet', 'Cassava, fresh', 'Sweet potatoes']:
    for label, yrs in [('All years 1991–2024, excl. 2008', [y for y in range(1991, 2025) if y != 2008]),
                       ('Measured-looking years only', [y for y in range(1991, 2025) if y not in range(2008, 2016)])]:
        x, y = rain_year.reindex(yrs), log_change[c].reindex(yrs)
        ok = x.notna() & y.notna()
        r, p = stats.pearsonr(x[ok], y[ok])
        rows.append({'crop': c, 'sample': label, 'r': r, 'p': p, 'n': int(ok.sum())})
resp = pd.DataFrame(rows).pivot(index='crop', columns='sample', values=['r', 'p'])
resp.round(2)

**Finding:** the response is patchy. **Sorghum** production does track rainfall (r = 0.37, and 0.54 when the projected years are left out, both significant), which fits a drought-sensitive crop grown mainly in the drier north and northeast. Beans and cassava show weak positive links that are not significant. **Maize, Uganda's main staple and most traded grain, shows no relationship at all** (r = 0.13), and millet and sweet potatoes none either.

**Conclusion for this project:** with a structural break in 2008, projected values in 2013–2015 and implausible jumps around 2018–2019, FAOSTAT's Uganda series can't support a reliable estimate of how rainfall affects harvests. Only sorghum carries a usable weather signal. Published analyses that use these series uncritically should be read with caution. This project instead uses **market prices**, which are collected every month and respond to real supply shocks (notebook 02). A second independent check, satellite vegetation indices over cropland, is a natural next step.